In [1]:
import os

import dotenv
from langchain_openai import ChatOpenAI

dotenv.load_dotenv()
os.environ["base_url"]


model = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

# 链的本质：一个 Runnable

**链就是一个实现了 `Runnable[Input, Output]` 接口的对象**

`|` 对应 `Runnable.__or__`（不是 `__add__`）：每次 `|` 都返回一个**新的 Runnable**（具体是 `RunnableSequence`），所以能无限链式拼接。`RunnableSequence`  在类层次上直接继承自 `Runnable`

In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import Runnable, RunnableSequence

prompt = PromptTemplate(
    template="讲一个关于{topic}的笑话，以台湾大学教机器学习的李宏毅老师的风格"
)
parser = StrOutputParser()

In [3]:
print(f"{isinstance(prompt,Runnable)=}  {isinstance(parser,Runnable)=}")
print(
    f"{issubclass(StrOutputParser,Runnable)=}  {issubclass(PromptTemplate,Runnable)=}"
)

isinstance(prompt,Runnable)=True  isinstance(parser,Runnable)=True
issubclass(StrOutputParser,Runnable)=True  issubclass(PromptTemplate,Runnable)=True


In [4]:
chain = prompt | model | parser

print(f"{type(chain)=} \n{chain=} \n{issubclass(type(chain),RunnableSequence)=})")

type(chain)=<class 'langchain_core.runnables.base.RunnableSequence'> 
chain=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='讲一个关于{topic}的笑话，以台湾大学教机器学习的李宏毅老师的风格')
| ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6', 'langchain-openai': '1.7.2'}}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001E4186C4830>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001E4186C52B0>, root_client=<openai.OpenAI object at 0x000001E4186C4050>, root_async_client=<openai.AsyncOpenAI object at 0x000001E4186C5010>, model_name='deepseek-ai/DeepSeek-V4.1-Flash', model_kwargs={}, openai_api_key=SecretStr('**********'), openai_api_base='https://api.siliconflow.cn/v1', stream_chunk_timeout=120.0)
| StrOutputParser() 
issubclass(type(chain),RunnableSequence)=True)


In [5]:
start = {"topic": "公园象棋"}
print(chain.invoke(start))

好，各位同学，我们今天不讲 gradient descent，来讲一个公园象棋的笑话。

假设你要做一个象棋 AI。你要找一个 function，input 是棋盘，output 是下一步。你收集了十万局棋谱，train 一个很大的 model，loss 一直降，accuracy 99%。你觉得：嗯，这个应该可以挑战公园阿伯了。

你到公园摆好棋盘。阿伯第一手炮二平五，你的 model 说：OK，这个看过。  
阿伯第二手，先喝一口茶，再把炮拿起来，又放回去，说：“我想一下。”  
你的 input 里面没有“喝茶”这个 feature，整个 attention 就乱掉。

阿伯第三手，走了一步你从来没看过。你抗议：“阿伯，这不合棋谱！”  
阿伯说：“棋谱是 training data，不是 ground truth。我赢了才是 ground truth。”

你问阿伯：“你的 policy 怎么 train 的？”  
阿伯说：“我每天来公园，赢了 reward 是隔壁阿嬷说‘水啦’，输了 reward 是‘我刚刚让你的’。这叫 reinforcement learning from human feedback，不过 feedback 只有我。”

你问：“那 learning rate 多少？”  
阿伯说：“我没有 learning rate，我有喝茶 rate。”

最后你输了。阿伯拍拍你说：“少年仔，你这个 model 太 overfitting。公园的 domain 有风、有烟、有阿嬷跳土风舞，你都没有 train。回去 data augmentation 一下。”

所以各位同学，machine learning 的尽头不是 AGI，是公园阿伯。  
好，那这个笑话的 loss 我们就不 backpropagate 了，下课。


# RunnableParallel


`RunnableParallel` 使用 `asyncio.gather` 来同时运行可运行对象

`ainvoke` 会沿链调用每个组件的 `ainvoke`。当执行到 `RunnablePassthrough.assign`（内部是 `RunnableParallel`）时，它会对 `translation1`、`translation2` 两个分支调用 `asyncio.gather`，**并发**执行 qwen 和 kimi 两次翻译，而不是串行。

In [6]:
deepseek_model = ChatOpenAI(
    model=os.environ["model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

qwen_model = ChatOpenAI(
    model=os.environ["secondary_model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

kimi_model = ChatOpenAI(
    model=os.environ["tertiary_model_name"],
    base_url=os.environ["base_url"],
    api_key=os.environ["api_key"],
)

RunnablePassthrough 接收输入并将其原样输出；LCEL 体系中的“无操作节点”，用于在流水线中透传输入或保留上下文，也可以用于向输出中添加键

In [16]:
from langchain_core.runnables import RunnablePassthrough

poem = "浔阳江头夜送客，枫叶荻花秋瑟瑟"
poem_template = PromptTemplate(template="请将下面的诗句翻译成越南语：{poem}")
judge_template = PromptTemplate(
    template="""
    分析下面两种对中文诗句：{poem}的翻译质量，判断哪种翻译更好，并给出理由。
    翻译1: {translation1},
    翻译2: {translation2}
"""
)

kimi_chain_handler = poem_template | kimi_model | parser
qwen_chain_handler = poem_template | qwen_model | parser
deepseek_chain_summary = judge_template | deepseek_model | parser

# RunnablePassthrough.assign 让第一个变量poem传递给
# kimi_chain_handler和qwen_chain_handler，得到两个翻译结果，
# 然后将这两个翻译结果传递给deepseek_chain_summary进行质量判断。

map_chain = RunnablePassthrough.assign(
    translation1=qwen_chain_handler,
    translation2=kimi_chain_handler,
) | RunnablePassthrough.assign(
    judge=deepseek_chain_summary,
)


In [17]:
result = await map_chain.ainvoke({"poem": poem})

In [ ]:
print(type(result), result.keys())

<class 'dict'> dict_keys(['poem', 'translation1', 'translation2', 'judge'])


In [22]:
from IPython.display import Markdown, display

display(
    Markdown(f"""
## 原句
{result["poem"]}

## 翻译1(Qwen)
{result["translation1"]}

## 翻译2(Kimi)
{result["translation2"]}

## 评判(DeepSeek)
{result["judge"]}
""")
)


## 原句
浔阳江头夜送客，枫叶荻花秋瑟瑟

## 翻译1(Qwen)


可译为：

**Đầu sông Tuân Dương, đêm tiễn khách,  
Lá phong hoa sậy xào xạc gió thu.**

更贴近诗意节奏的说法：

**Đêm đầu sông Tuân Dương tiễn khách,  
Lá phong hoa sậy xào xạc thu phong.**

## 翻译2(Kimi)
这是白居易《琵琶行》的开篇名句。以下是比较贴近原意的越南语翻译：

**Đêm đầu sông Tầm Dương tiễn khách,**
**Lá phong hoa địch, thu xao xác lạnh.**

### 词汇对照说明：
- **Tầm Dương** (浔阳)：今江西九江一带，长江流经的一段。
- **Lá phong** (枫叶)：枫树叶。
- **Hoa địch** (荻花)：荻草的花（一种类似芦苇的水草之花）。
- **Xao xác lạnh**：形容秋风萧瑟、寒意袭人，对应“瑟瑟”的凄凉秋声与秋意。

如果需要更口语化的解释版本，也可以译为：  
*Đêm ở bến sông Tầm Dương tiễn vị khách, lá phong và hoa địch trong gió thu xào xạc lạnh buốt.*

## 评判(DeepSeek)
综合来看，**翻译2更好**，尤其是在“贴近原意”和“解释完整度”上明显胜出。翻译1在越南语的可读性、诗意节奏上有优点，但有专名错误和个别词义不够精确的问题。

逐点比较：

1. **专名“浔阳”**  
   - 翻译2作 **Tầm Dương**，正确。  
   - 翻译1作 **Tuân Dương**，这是硬伤；“浔”的汉越音应为 **tầm**，不是 **tuân**。  
   所以仅此一点，翻译2更可靠。

2. **“江头”的理解**  
   “江头”指江边、渡口一带，不宜理解为“江的源头”。两篇主译文都用了 **đầu sông**，都不够理想。  
   但翻译2后面补充了 **bến sông**，即“江边/渡口”，比翻译1只作 **đầu sông** 更准确。

3. **“荻花”**  
   - 翻译2作 **hoa địch**，对应“荻”的汉越音 **địch**，更精确，而且有注释说明。  
   - 翻译1作 **hoa sậy**，自然易懂，但“荻”与“sậy”并不完全等同，属于近似替换。  
   若面向普通读者，其实 **hoa lau** 可能比 **hoa địch** 更自然；但就“贴近原意”而言，翻译2更严谨。

4. **“秋瑟瑟”**  
   “瑟瑟”兼有秋风萧瑟、草木作响、寒意凄凉之意。  
   - 翻译1“**xào xạc gió thu / xào xạc thu phong**”主要译出风声、秋意，较顺口，但寒意层次不够。  
   - 翻译2“**thu xao xác lạnh**”兼顾了“萧瑟”与“寒”，虽有稍生硬之处，但意义更完整。其补充译文“**trong gió thu xào xạc lạnh buốt**”更把风声和寒意都补足了。

5. **诗意与流畅度**  
   翻译1的“**Lá phong hoa sậy xào xạc gió thu**”更顺口，越南语读者更容易接受。  
   翻译2主句“**Lá phong hoa địch, thu xao xác lạnh**”因为 **hoa địch** 生僻、**thu xao xác lạnh** 略拗，诗化流畅度稍弱；但它用注释和口语化译文弥补了理解障碍。

**结论：**  
如果只追求顺口和诗感，翻译1有优势；但若判断翻译质量，尤其是准确性、名物对应、原意层次和注释完整度，**翻译2整体更好**。翻译1最大的问题是 **Tuân Dương** 错误；翻译2虽主句略生硬，但基本正确，并提供了更准确的补充译文。

较理想的合并译法可以是：

**Đêm ở bến sông Tầm Dương tiễn khách,  
Lá phong hoa lau xào xạc trong gió thu lạnh.**
